# Low-Float Reversal Short Watchlist — August 17, 2026

## tl;dr

- **IVF** is the cleanest code-comparable setup: the price failed hard, the catalyst headline is weaker than it first appears, and an effective resale prospectus covers up to 20 million shares versus roughly 2.5 million shares outstanding.
- **TRUG** crosses the alert threshold only after rolling its Friday-after-close 10-Q into Monday's session. Its business improvement is real, so it needs continued failure below VWAP rather than a blind short.
- **OSRH** has strong supply risk but still has intact momentum and an unverified float gate. **IPST** and **WETO** can score well mechanically but are blocked by missing point-in-time catalysts; **UCL** lacks the supply-risk asymmetry and no longer passes the live return gate.
- These are research candidates, not execution-ready trades. Borrow, locate cost, spread, Rule 201 handling, and live inventory were not available.

## Context & Methods

This notebook applies the repository's default detector and score philosophy to the U.S. low-float movers visible on August 17, 2026. The default detector requires a $0.25–$20 price, no more than 20 million float shares, at least a 50% Day-0 move, at least 5x relative volume, and at least $1 million in dollar volume.

The proxy score reproduces the checked-in weights: 48% momentum failure, 20% catalyst fragility, 20% supply risk, and 12% exhaustion, less any observed halt penalty. Yahoo's one-minute bars are used to approximate cumulative VWAP; filing-derived catalyst and supply inputs are documented below. Scores are evidence ratings, not return probabilities.

**Timing assumption:** filings accepted after the Friday regular-session close are normalized to Monday's next regular session for the research view. The current implementation compares calendar dates and may otherwise treat TRUG and OSRH as having no Day-0 catalyst.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)


def clamp(value, low=0.0, high=1.0):
    return max(low, min(high, value))

## Data

Market inputs were captured from Yahoo Finance one-minute and daily chart endpoints at approximately 3:31 p.m. EDT on August 17, 2026. Float denominators use issuer-reported shares outstanding when that number itself is below the detector cap; this is conservative because float cannot exceed outstanding shares. OSRH and UCL remain unverified because their outstanding/ADS-equivalent counts exceed 20 million.

Primary filing sources: [IVF 8-K exhibit](https://www.sec.gov/Archives/edgar/data/1417926/000149315226038648/ex99-1.htm), [IVF 10-Q](https://www.sec.gov/Archives/edgar/data/1417926/000149315226038528/form10-q.htm), [IVF 424B3](https://www.sec.gov/Archives/edgar/data/1417926/000149315226036567/form424b3.htm), [TRUG 10-Q](https://www.sec.gov/Archives/edgar/data/1857086/000149315226038488/form10-q.htm), [OSRH 10-Q](https://www.sec.gov/Archives/edgar/data/1840425/000121390026089904/ea0301514-10q_osrhealth.htm), [OSRH proxy](https://www.sec.gov/Archives/edgar/data/1840425/000121390026078706/ea0296871-02.htm), [IPST S-3](https://www.sec.gov/Archives/edgar/data/1788230/000149315226026907/forms-3.htm), [UCL 6-K exhibit](https://www.sec.gov/Archives/edgar/data/1775898/000121390026090488/ea030236001ex99-1.htm), [UCL 20-F](https://www.sec.gov/Archives/edgar/data/1775898/000121390026035481/ea0282596-20f_ucloudlink.htm), [WETO 424B5](https://www.sec.gov/Archives/edgar/data/1941158/000121390026075590/ea0297083-424b5_wetour.htm), and [WETO August 12 6-K](https://www.sec.gov/Archives/edgar/data/1941158/000121390026088400/ea0301733-6k_wetour.htm).

In [2]:
snapshot = [
    dict(
        symbol="IVF",
        cutoff_et="2026-08-17 15:31:22 EDT",
        price=1.5001,
        prev_close=0.9540,
        session_high=2.8000,
        volume=111_689_726,
        avg20_volume=75_200,
        denom_shares=2_506_969,
        denom_basis="Shares outstanding, Aug. 14",
        vwap=2.2600,
        lower_high_ratio=0.50,
        failed_vwap_reclaims=3,
        opening_range_breakdown=True,
        volume_fade_ratio=0.052,
        supply_score=1.00,
        materiality=0.35,
        novelty=1.00,
        promotional_risk=0,
        catalyst_confidence=0.825,
        halt_count=0,
        price_gate="pass",
        float_gate="pass",
        catalyst_timing="Monday premarket 8-K",
    ),
    dict(
        symbol="TRUG",
        cutoff_et="2026-08-17 15:31:29 EDT",
        price=1.5799,
        prev_close=0.9695,
        session_high=1.8199,
        volume=61_076_879,
        avg20_volume=178_195,
        denom_shares=1_923_707,
        denom_basis="Class A + B outstanding, Aug. 12",
        vwap=1.6136,
        lower_high_ratio=0.25,
        failed_vwap_reclaims=17,
        opening_range_breakdown=True,
        volume_fade_ratio=0.086,
        supply_score=0.81,
        materiality=0.35,
        novelty=1.00,
        promotional_risk=0,
        catalyst_confidence=0.825,
        halt_count=0,
        price_gate="pass",
        float_gate="pass",
        catalyst_timing="Friday after close; rolled to Monday",
    ),
    dict(
        symbol="OSRH",
        cutoff_et="2026-08-17 15:31:30 EDT",
        price=0.6900,
        prev_close=0.3160,
        session_high=0.8000,
        volume=345_635_593,
        avg20_volume=696_605,
        denom_shares=35_118_692,
        denom_basis="Shares outstanding, Aug. 10",
        vwap=0.5609,
        lower_high_ratio=0.50,
        failed_vwap_reclaims=9,
        opening_range_breakdown=False,
        volume_fade_ratio=0.948,
        supply_score=0.98,
        materiality=0.35,
        novelty=1.00,
        promotional_risk=0,
        catalyst_confidence=0.825,
        halt_count=0,
        price_gate="pass",
        float_gate="unverified",
        catalyst_timing="Friday after close; rolled to Monday",
    ),
    dict(
        symbol="IPST",
        cutoff_et="2026-08-17 15:31:29 EDT",
        price=7.2363,
        prev_close=2.2001,
        session_high=9.6600,
        volume=97_947_696,
        avg20_volume=62_365,
        denom_shares=721_578,
        denom_basis="Shares outstanding, Jun. 1",
        vwap=7.9601,
        lower_high_ratio=0.50,
        failed_vwap_reclaims=34,
        opening_range_breakdown=True,
        volume_fade_ratio=0.049,
        supply_score=1.00,
        materiality=0,
        novelty=0,
        promotional_risk=0,
        catalyst_confidence=0,
        halt_count=0,
        price_gate="pass",
        float_gate="pass",
        catalyst_timing="No same-day issuer filing or release found",
    ),
    dict(
        symbol="UCL",
        cutoff_et="2026-08-17 15:31:23 EDT",
        price=0.6675,
        prev_close=0.7300,
        session_high=1.0000,
        volume=89_882_712,
        avg20_volume=20_990,
        denom_shares=25_849_968,
        denom_basis="Class A ADS-equivalent upper bound",
        vwap=0.9229,
        lower_high_ratio=0.50,
        failed_vwap_reclaims=1,
        opening_range_breakdown=True,
        volume_fade_ratio=0.013,
        supply_score=0,
        materiality=0.65,
        novelty=1.00,
        promotional_risk=0,
        catalyst_confidence=0.825,
        halt_count=0,
        price_gate="pass",
        float_gate="unverified",
        catalyst_timing="Monday premarket 6-K",
    ),
    dict(
        symbol="WETO",
        cutoff_et="2026-08-17 15:31:31 EDT",
        price=26.6500,
        prev_close=8.2200,
        session_high=29.4991,
        volume=27_938_252,
        avg20_volume=4_041_287,
        denom_shares=1_078_000,
        denom_basis="Approx. post-split outstanding",
        vwap=18.3597,
        lower_high_ratio=0.50,
        failed_vwap_reclaims=1,
        opening_range_breakdown=False,
        volume_fade_ratio=0.406,
        supply_score=1.00,
        materiality=0,
        novelty=0,
        promotional_risk=0,
        catalyst_confidence=0,
        halt_count=0,
        price_gate="fail > $20",
        float_gate="pass",
        catalyst_timing="No same-day issuer filing or release found",
    ),
]
df = pd.DataFrame(snapshot)
df

,symbol,cutoff_et,price,prev_close,session_high,volume,avg20_volume,denom_shares,denom_basis,vwap,lower_high_ratio,failed_vwap_reclaims,opening_range_breakdown,volume_fade_ratio,supply_score,materiality,novelty,promotional_risk,catalyst_confidence,halt_count,price_gate,float_gate,catalyst_timing
0,IVF,2026-08-17 15:31:22 EDT,1.5001,0.9540,2.8000,111689726,75200,2506969,"Shares outstanding, Aug. 14",2.2600,0.50,3,True,0.052,1.00,0.35,1.0,0,0.825,0,pass,pass,Monday premarket 8-K
1,TRUG,2026-08-17 15:31:29 EDT,1.5799,0.9695,1.8199,61076879,178195,1923707,"Class A + B outstanding, Aug. 12",1.6136,0.25,17,True,0.086,0.81,0.35,1.0,0,0.825,0,pass,pass,Friday after close; rolled to Monday
2,OSRH,2026-08-17 15:31:30 EDT,0.6900,0.3160,0.8000,345635593,696605,35118692,"Shares outstanding, Aug. 10",0.5609,0.50,9,False,0.948,0.98,0.35,1.0,0,0.825,0,pass,unverified,Friday after close; rolled to Monday
3,IPST,2026-08-17 15:31:29 EDT,7.2363,2.2001,9.6600,97947696,62365,721578,"Shares outstanding, Jun. 1",7.9601,0.50,34,True,0.049,1.00,0.00,0.0,0,0.000,0,pass,pass,No same-day issuer filing or release found
4,UCL,2026-08-17 15:31:23 EDT,0.6675,0.7300,1.0000,89882712,20990,25849968,Class A ADS-equivalent upper bound,0.9229,0.50,1,True,0.013,0.00,0.65,1.0,0,0.825,0,pass,unverified,Monday premarket 6-K
5,WETO,2026-08-17 15:31:31 EDT,26.6500,8.2200,29.4991,27938252,4041287,1078000,Approx. post-split outstanding,18.3597,0.50,1,False,0.406,1.00,0.00,0.0,0,0.000,0,fail > $20,pass,No same-day issuer filing or release found


In [3]:
df["return_pct"] = (df.price / df.prev_close - 1) * 100
df["close_off_high_pct"] = (df.session_high - df.price) / df.session_high * 100
df["vwap_distance_pct"] = (df.price / df.vwap - 1) * 100
df["rvol_x"] = df.volume / df.avg20_volume
df["turnover_x"] = df.volume / df.denom_shares


def momentum_failure(row):
    components = {
        "below_vwap": clamp(-row.vwap_distance_pct / 8),
        "off_high": clamp(row.close_off_high_pct / 35),
        "lower_highs": row.lower_high_ratio,
        "failed_reclaims": clamp(row.failed_vwap_reclaims / 3),
        "opening_breakdown": float(row.opening_range_breakdown),
        "volume_fade": clamp((1 - row.volume_fade_ratio) / 0.75),
    }
    return 100 * (
        components["below_vwap"] * 0.22
        + components["off_high"] * 0.18
        + components["lower_highs"] * 0.18
        + components["failed_reclaims"] * 0.17
        + components["opening_breakdown"] * 0.15
        + components["volume_fade"] * 0.10
    )


df["momentum_failure"] = df.apply(momentum_failure, axis=1)
df["catalyst_fragility"] = 100 * df.apply(
    lambda r: clamp(
        (1 - r.materiality) * 0.45 + (1 - r.novelty) * 0.20 + r.promotional_risk * 0.35
    ),
    axis=1,
)
df["exhaustion"] = 100 * df.apply(
    lambda r: clamp(
        (r.turnover_x / 5) * 0.55
        + clamp((r.rvol_x - 5) / 20) * 0.25
        + clamp((r.close_off_high_pct / 100) / 0.4) * 0.20
    ),
    axis=1,
)
df["proxy_score"] = df.apply(
    lambda r: (
        clamp(
            (
                r.momentum_failure * 0.48
                + r.catalyst_fragility * 0.20
                + r.supply_score * 100 * 0.20
                + r.exhaustion * 0.12
                - min(r.halt_count * 2.5, 12.5)
            )
            / 100
        )
        * 100
    ),
    axis=1,
)


def gate_status(row):
    if row.catalyst_confidence < 0.20:
        return "BLOCKED — catalyst"
    if row.float_gate != "pass" or row.price_gate != "pass" or row.return_pct < 50:
        return "EXCLUDE / verify gate"
    if row.proxy_score >= 78:
        return "HIGH-CONVICTION RESEARCH"
    if row.proxy_score >= 62:
        return "ALERT RESEARCH"
    return "WATCH"


df["status"] = df.apply(gate_status, axis=1)
ranked = df.sort_values("proxy_score", ascending=False)
ranked[
    [
        "symbol",
        "proxy_score",
        "status",
        "return_pct",
        "close_off_high_pct",
        "vwap_distance_pct",
        "rvol_x",
        "turnover_x",
        "momentum_failure",
        "supply_score",
        "catalyst_confidence",
    ]
].round(1)

,symbol,proxy_score,status,return_pct,close_off_high_pct,vwap_distance_pct,rvol_x,turnover_x,momentum_failure,supply_score,catalyst_confidence
3,IPST,86.2,BLOCKED — catalyst,228.9,25.1,-9.1,1570.6,135.7,85.9,1.0,0.0
0,IVF,81.5,HIGH-CONVICTION RESEARCH,57.2,46.4,-33.6,1485.2,44.6,91.0,1.0,0.8
1,TRUG,62.4,ALERT RESEARCH,63.0,13.2,-2.1,342.8,31.7,59.0,0.8,0.8
5,WETO,58.2,BLOCKED — catalyst,224.2,9.7,45.2,6.9,25.9,27.6,1.0,0.0
2,OSRH,53.7,EXCLUDE / verify gate,118.4,13.8,23.0,496.2,9.8,33.8,1.0,0.8
4,UCL,50.5,EXCLUDE / verify gate,-8.6,33.2,-27.7,4282.2,3.5,78.8,0.0,0.8


## Results

### IVF — highest-priority research candidate

The tape supplied nearly every failure input: about 46% off the regular-session high, roughly 34% below cumulative VWAP, an opening-range breakdown, failed VWAP reclaims, and extreme turnover. The press-release headline says the clinic platform was profitable before corporate/public-company costs, but consolidated adjusted EBITDA was approximately negative $1.0 million versus negative $0.6 million in the prior-year quarter; reported net income included an approximately $2.5 million remeasurement gain. The effective 424B3 covers up to 20 million resale shares, about 8x the roughly 2.5 million shares outstanding reported August 14.

### TRUG — alert threshold, but a stronger underlying catalyst

The next-session-normalized proxy reaches the 62 alert line as price slips below VWAP. Unlike IVF, the Q2 improvement is not merely cosmetic: revenue rose about 34%, gross profit nearly doubled, and the operating loss narrowed sharply. The bearish asymmetry instead comes from 31x illustrative turnover, a recent 1-for-10 reverse split, 100 million authorized Class A shares versus roughly 1.9 million total shares outstanding, and reset/anti-dilution features on Series A preferred stock. This is a confirmation setup, not a short-into-strength setup.

### OSRH — supply risk is strong; the chart and float gate are not ready

Q2 net sales fell 72% while gross profit rose 165% from a tiny base; the company reported $1.5 million of cash. The 10-Q says OSRH expects to keep using its $80 million White Lion equity line through December 31 and plans an ATM. Yet the price remained roughly 23% above VWAP at the cutoff. The issuer reported 35.1 million shares outstanding and the proxy's officer/director group owned 13.1 million, leaving about 22.0 million before considering any other restrictions—just above the tool's 20 million float cap.

### Gated names

IPST's mechanical score is high, but no August 17 issuer release or filing was found; the minimum catalyst-confidence gate therefore blocks it. WETO is also blocked by no same-day catalyst, remained well above VWAP, and had moved beyond the default $20 ceiling. UCL completed a spectacular reversal after a $2 million repurchase announcement, but its live return fell below the detector threshold, its Class A ADS-equivalent count is above 20 million before affiliate adjustments, and no comparable dilution stack was identified.

In [4]:
# Validation checks for the high-impact calculations and gates.
assert abs(df.loc[df.symbol.eq("IVF"), "turnover_x"].iloc[0] - 44.6) < 0.2
assert df.loc[df.symbol.eq("IVF"), "proxy_score"].iloc[0] >= 78
assert 62 <= df.loc[df.symbol.eq("TRUG"), "proxy_score"].iloc[0] < 78
assert df.loc[df.symbol.eq("IPST"), "catalyst_confidence"].iloc[0] == 0
assert df.loc[df.symbol.eq("WETO"), "price_gate"].iloc[0] != "pass"
assert df.loc[df.symbol.eq("UCL"), "return_pct"].iloc[0] < 0
assert (df.proxy_score.between(0, 100)).all()
"All validation checks passed."

'All validation checks passed.'

## Takeaways

1. **IVF is the only high-conviction research candidate in this snapshot.** It combines a completed technical failure with immediate registered supply and a headline-to-economics mismatch. It may already be late after the fade, so execution still requires a controlled setup rather than chasing.
2. **TRUG is the next name to monitor.** It sits around the alert line, but the business catalyst is materially stronger; require persistence below VWAP or a failed reclaim.
3. **OSRH belongs on the watchlist, not the short list yet.** Supply is abundant, but momentum remained intact and the float gate needs a proper vendor value.
4. **Keep the gates.** IPST and WETO demonstrate why a high raw score is not enough when the point-in-time catalyst is missing.

### Caveats and assumptions

- The market snapshot is intraday, not a closing print, and Yahoo one-minute bars are a proxy for the production feed.
- Supply scores are transparent filing-based approximations using the repository's term weights; they are not vendor-normalized production observations.
- No official halt feed, broker borrow/locate inventory, fee, spread, or Rule 201 execution state was available. Zero halt penalties reflect no one-minute regular-session gaps for the leading names, not an official halt certification.
- Shorting low-float names can produce losses greater than the initial position size. This document is research, not personalized investment advice.